In [1]:
try:
    from google.colab import drive  # Colab only
except Exception:
    drive = None
import os as _os
if drive is not None and not _os.path.ismount("/content/drive"):
    drive.mount("/content/drive")  # [sanitized] optional; data paths are relative to DrugReview_ROOT

import sys
from pathlib import Path
import os
import pandas as pd

DrugReview_ROOT = Path(".")
sys.path.append(str(DrugReview_ROOT))



RAW_DATA_FILE = "PureText/drugsCom_mood_anxiety_with_ai_labels_mini.csv"
DATA_PATH = os.path.join(DrugReview_ROOT, "data", RAW_DATA_FILE)  # or os.path.join(PROJECT_PATH, RAW_DATA_FILE)
df = pd.read_csv(DATA_PATH)

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
)

# ---------------------------
# Config
# ---------------------------
HUMAN_COL = "sentiment_5"         # human 5-class
AI_HARD_COL = "ai_sentiment_5"    # GPT hard 5-class
AI_RATING_COL = "ai_rating_10"    # GPT 1..10

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
LABEL2IDX = {lab:i for i,lab in enumerate(ORDER5)}
IDX2LABEL = {i:lab for lab,i in LABEL2IDX.items()}

# ---------------------------
# Helpers
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def rating10_to_sent5(r):
    if pd.isna(r):
        return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

def defensive_renorm(p, eps=1e-12):
    """Clip + renormalize rows to sum to 1 (numerical tolerance)."""
    p = np.clip(p, eps, 1.0)
    row_sums = p.sum(axis=1, keepdims=True)
    return p / row_sums

def entropy_norm(p):
    """Normalized Shannon entropy in [0,1] (natural log)."""
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def multiclass_brier(p, y_idx):
    """Mean sum_k (p_k - 1[y=k])^2."""
    n, K = p.shape
    y_onehot = np.zeros((n, K), dtype=float)
    y_onehot[np.arange(n), y_idx] = 1.0
    return np.mean(np.sum((p - y_onehot)**2, axis=1))

# ---------------------------
# 1) Normalize labels
# ---------------------------
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)

# rating-derived 5-class from GPT rating_10
df["ai_sent5_from_rating10"] = df[AI_RATING_COL].apply(rating10_to_sent5)

# ---------------------------
# 2) Build probability matrix + argmax label
# ---------------------------
missing = [c for c in PROB_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing probability columns: {missing}"

p_raw = df[PROB_COLS].astype(float).to_numpy()
p = defensive_renorm(p_raw)

df["ai_prob_argmax"] = np.argmax(p, axis=1)
df["ai_sent5_from_probs"] = df["ai_prob_argmax"].map(IDX2LABEL)

df["ai_pmax"] = p.max(axis=1)
df["ai_entropy_norm"] = entropy_norm(p)

# ---------------------------
# 3) HARD-LABEL AGREEMENT (human vs different AI hard labels)
# ---------------------------
def hard_agreement(y_true, y_pred, name):
    mask = y_true.notna() & y_pred.notna()
    yt = y_true[mask].astype(str)
    yp = y_pred[mask].astype(str)

    acc = accuracy_score(yt, yp)
    qwk = cohen_kappa_score(yt, yp, labels=ORDER5, weights="quadratic")

    cm = confusion_matrix(yt, yp, labels=ORDER5)
    cm_df = pd.DataFrame(cm, index=[f"T:{c}" for c in ORDER5], columns=[f"P:{c}" for c in ORDER5])

    print(f"\n=== {name} ===")
    print(f"N = {len(yt)}")
    print(f"Exact agreement (accuracy): {acc:.4f}")
    print(f"Quadratic weighted kappa:   {qwk:.4f}")
    print("\nConfusion matrix (counts):")
    display(cm_df)
    print("\nClassification report:")
    print(classification_report(yt, yp, labels=ORDER5, zero_division=0))

# A) Human vs GPT hard label
hard_agreement(df[HUMAN_COL], df[AI_HARD_COL], "Human sentiment_5 vs AI hard ai_sentiment_5")

# B) Human vs GPT rating10-binned label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_rating10"], "Human sentiment_5 vs AI (ai_rating_10 -> 5-class)")

# C) Human vs Prob-argmax label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_probs"], "Human sentiment_5 vs AI (prob argmax)")

# D) Internal consistency: GPT hard vs prob-argmax
hard_agreement(df[AI_HARD_COL], df["ai_sent5_from_probs"], "AI hard ai_sentiment_5 vs AI prob-argmax")

# ---------------------------
# 4) SOFT-LABEL METRICS (human vs probability vector)
# ---------------------------
mask_soft = df[HUMAN_COL].notna()
y_true = df.loc[mask_soft, HUMAN_COL].map(LABEL2IDX).astype(int).to_numpy()
p_soft = p[mask_soft.to_numpy()]

# Prob assigned to the human class
p_true = p_soft[np.arange(len(y_true)), y_true]
mean_p_true = float(np.mean(p_true))

# NLL of the human class (cross-entropy on observed label)
nll = float(np.mean(-np.log(np.clip(p_true, 1e-12, 1.0))))

# Multiclass Brier
brier = float(multiclass_brier(p_soft, y_true))

# Ordinal expected class index + error
idx = np.arange(5)
exp_idx = p_soft @ idx
mae = float(np.mean(np.abs(exp_idx - y_true)))
rmse = float(np.sqrt(np.mean((exp_idx - y_true)**2)))

# Expected absolute ordinal error (W1 to one-hot along ordinal axis)
# (equivalent here to E[|K - y|] under p)
w1 = float(np.mean(np.sum(p_soft * np.abs(idx[None, :] - y_true[:, None]), axis=1)))

print("\n=== Soft-label diagnostics (Human vs AI prob vector) ===")
print(f"N = {len(y_true)}")
print(f"Mean P(true human class): {mean_p_true:.4f}")
print(f"NLL (cross-entropy):     {nll:.4f}")
print(f"Brier (multiclass):      {brier:.4f}")
print(f"Ordinal MAE (E[idx]-y):  {mae:.4f}")
print(f"Ordinal RMSE:            {rmse:.4f}")
print(f"Expected |ordinal error|:{w1:.4f}")

# ---------------------------
# 5) Stratify agreement by confidence / uncertainty (optional but very informative)
# ---------------------------
tmp = df.loc[mask_soft, [HUMAN_COL, "ai_sent5_from_probs", "ai_pmax", "ai_entropy_norm"]].copy()
tmp["correct_prob_argmax"] = (tmp[HUMAN_COL] == tmp["ai_sent5_from_probs"])

# bins for pmax and entropy
tmp["pmax_bin"] = pd.cut(tmp["ai_pmax"], bins=[0, .4, .6, .8, .9, 1.0], include_lowest=True)
tmp["ent_bin"]  = pd.cut(tmp["ai_entropy_norm"], bins=[0, .2, .4, .6, .8, 1.0], include_lowest=True)

summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})

print("\nAgreement vs p_max bins (Human vs prob-argmax):")
display(summary_pmax)

print("\nAgreement vs entropy bins (Human vs prob-argmax):")
display(summary_ent)


=== Human sentiment_5 vs AI hard ai_sentiment_5 ===
N = 28755
Exact agreement (accuracy): 0.5457
Quadratic weighted kappa:   0.8121

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,2944,869,114,19,7
T:negative,466,859,264,36,10
T:neutral,190,862,964,235,31
T:positive,73,417,1798,2720,690
T:very_positive,48,233,1029,5671,8206



Classification report:
               precision    recall  f1-score   support

very_negative       0.79      0.74      0.77      3953
     negative       0.27      0.53      0.35      1635
      neutral       0.23      0.42      0.30      2282
     positive       0.31      0.48      0.38      5698
very_positive       0.92      0.54      0.68     15187

     accuracy                           0.55     28755
    macro avg       0.50      0.54      0.50     28755
 weighted avg       0.69      0.55      0.58     28755


=== Human sentiment_5 vs AI (ai_rating_10 -> 5-class) ===
N = 28755
Exact agreement (accuracy): 0.5457
Quadratic weighted kappa:   0.8121

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,2944,869,114,19,7
T:negative,466,859,264,36,10
T:neutral,190,862,964,235,31
T:positive,73,417,1798,2720,690
T:very_positive,48,233,1029,5671,8206



Classification report:
               precision    recall  f1-score   support

very_negative       0.79      0.74      0.77      3953
     negative       0.27      0.53      0.35      1635
      neutral       0.23      0.42      0.30      2282
     positive       0.31      0.48      0.38      5698
very_positive       0.92      0.54      0.68     15187

     accuracy                           0.55     28755
    macro avg       0.50      0.54      0.50     28755
 weighted avg       0.69      0.55      0.58     28755


=== Human sentiment_5 vs AI (prob argmax) ===
N = 28755
Exact agreement (accuracy): 0.5713
Quadratic weighted kappa:   0.8053

Confusion matrix (counts):


...


Classification report:
               precision    recall  f1-score   support

very_negative       0.74      0.99      0.85      3721
     negative       0.60      0.60      0.60      3240
      neutral       0.93      0.18      0.30      4169
     positive       0.65      0.73      0.69      8681
very_positive       0.76      0.85      0.81      8944

     accuracy                           0.71     28755
    macro avg       0.74      0.67      0.65     28755
 weighted avg       0.73      0.71      0.68     28755


=== Soft-label diagnostics (Human vs AI prob vector) ===
N = 28755
Mean P(true human class): 0.4241
NLL (cross-entropy):     1.3267
Brier (multiclass):      0.5493
Ordinal MAE (E[idx]-y):  0.6961
Ordinal RMSE:            0.8538
Expected |ordinal error|:0.8325

Agreement vs p_max bins (Human vs prob-argmax):


/tmp/ipython-input-1095427977.py:172: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
/tmp/ipython-input-1095427977.py:173: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})


,count,acc
pmax_bin,,
"(-0.001, 0.4]",11150,0.342870
"(0.4, 0.6]",14085,0.681789
"(0.6, 0.8]",2630,0.843726
"(0.8, 0.9]",735,0.884354
"(0.9, 1.0]",155,0.851613



Agreement vs entropy bins (Human vs prob-argmax):


,count,acc
ent_bin,,
"(-0.001, 0.2]",155,0.851613
"(0.2, 0.4]",2665,0.861163
"(0.4, 0.6]",10912,0.758614
"(0.6, 0.8]",9389,0.427309
"(0.8, 1.0]",5633,0.303568


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score

# --- helpers ---
LABELS_5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]

def normalize_5class(x):
    """Make ai_sentiment_5 robust to casing / spaces / hyphens."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    # common variants (optional)
    mapping = {
        "verynegative": "very_negative",
        "verypositive": "very_positive",
        "vneg": "very_negative",
        "vpos": "very_positive",
    }
    return mapping.get(s, s)

def rating10_to_5class(r):
    """Map 1–10 rating to 5 ordinal bins: 1-2,3-4,5-6,7-8,9-10."""
    if pd.isna(r):
        return pd.NA
    try:
        rr = int(float(r))
    except Exception:
        return pd.NA
    if rr < 1 or rr > 10:
        return pd.NA
    if rr <= 2:
        return "very_negative"
    elif rr <= 4:
        return "negative"
    elif rr <= 6:
        return "neutral"
    elif rr <= 8:
        return "positive"
    else:
        return "very_positive"

# --- main ---
# df = ...  # your dataframe containing ai_sentiment_5 and ai_rating_10

df = df.copy()
df["ai_sentiment_5_norm"] = df["ai_sentiment_5"].apply(normalize_5class)
df["ai_sent5_from_rating10"] = df["ai_rating_10"].apply(rating10_to_5class)

sub = df[["ai_sentiment_5_norm", "ai_sent5_from_rating10"]].dropna()

y1 = sub["ai_sentiment_5_norm"]
y2 = sub["ai_sent5_from_rating10"]

acc = accuracy_score(y1, y2)
kappa_q = cohen_kappa_score(y1, y2, labels=LABELS_5, weights="quadratic")
cm = confusion_matrix(y1, y2, labels=LABELS_5)

print(f"N = {len(sub)}")
print(f"Exact agreement (accuracy): {acc:.4f}")
print(f"Quadratic weighted kappa:   {kappa_q:.4f}\n")

print("Confusion matrix (counts):")
print(pd.DataFrame(cm, index=[f"T:{c}" for c in LABELS_5], columns=[f"P:{c}" for c in LABELS_5]))
print("\nClassification report:")
print(classification_report(y1, y2, labels=LABELS_5, digits=2))

N = 28755
Exact agreement (accuracy): 1.0000
Quadratic weighted kappa:   1.0000

Confusion matrix (counts):
                 P:very_negative  P:negative  P:neutral  P:positive  \
T:very_negative             3721           0          0           0   
T:negative                     0        3240          0           0   
T:neutral                      0           0       4169           0   
T:positive                     0           0          0        8681   
T:very_positive                0           0          0           0   

                 P:very_positive  
T:very_negative                0  
T:negative                     0  
T:neutral                      0  
T:positive                     0  
T:very_positive             8944  

Classification report:
               precision    recall  f1-score   support

very_negative       1.00      1.00      1.00      3721
     negative       1.00      1.00      1.00      3240
      neutral       1.00      1.00      1.00      4169
     posit

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score

# ---------------------------
# REQUIREMENTS from your existing pipeline:
# df contains:
#   HUMAN_COL (sentiment_5)
#   AI_HARD_COL (ai_sentiment_5)
#   ai_sent5_from_probs (prob argmax label, 5-class)
#   ai_pmax (max prob)
# ---------------------------

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
VAL5 = {"very_negative": -2, "negative": -1, "neutral": 0, "positive": 1, "very_positive": 2}

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def label_distribution(s, order=ORDER5):
    s = s.dropna().astype(str)
    counts = s.value_counts().reindex(order).fillna(0).astype(int)
    props = counts / counts.sum() if counts.sum() > 0 else counts.astype(float)
    return pd.DataFrame({"count": counts, "prop": props})

def jensen_shannon(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return float(np.sqrt(0.5 * (kl_pm + kl_qm)))

def ordinal_error_summary(y_true, y_pred):
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str)
    yp = yp[m].astype(str)

    yv = yt.map(VAL5).to_numpy()
    pv = yp.map(VAL5).to_numpy()

    abs_err = np.abs(pv - yv)
    out = {
        "N": int(m.sum()),
        "mean_abs_ord_err": float(abs_err.mean()),
        "ord_rmse": float(np.sqrt(np.mean((pv - yv)**2))),
        "within_1_class": float((abs_err <= 1).mean()),
    }
    return out

def bootstrap_ci_acc_qwk(y_true, y_pred, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str).to_numpy()
    yp = yp[m].astype(str).to_numpy()
    n = len(yt)

    accs = np.empty(B, dtype=float)
    qwks = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        ytb, ypb = yt[idx], yp[idx]
        accs[b] = accuracy_score(ytb, ypb)
        qwks[b] = cohen_kappa_score(ytb, ypb, labels=ORDER5, weights="quadratic")

    def ci(x):
        return (float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975)))

    return {
        "N": n,
        "acc_mean": float(accs.mean()),
        "acc_ci95": ci(accs),
        "qwk_mean": float(qwks.mean()),
        "qwk_ci95": ci(qwks),
    }

def ece_from_pmax(y_true, y_pred, pmax, n_bins=10):
    y_true = np.asarray([norm5(x) for x in y_true])
    y_pred = np.asarray([norm5(x) for x in y_pred])
    pmax = np.asarray(pmax, dtype=float)

    m = pd.notna(y_true) & pd.notna(y_pred) & pd.notna(pmax)
    y_true = y_true[m]
    y_pred = y_pred[m]
    pmax = pmax[m]

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        sel = (pmax > lo) & (pmax <= hi) if i > 0 else (pmax >= lo) & (pmax <= hi)
        if sel.sum() == 0:
            rows.append([f"({lo:.1f},{hi:.1f}]", 0, np.nan, np.nan, np.nan])
            continue
        acc_b = float((y_true[sel] == y_pred[sel]).mean())
        conf_b = float(pmax[sel].mean())
        w = float(sel.mean())
        ece += w * abs(acc_b - conf_b)
        rows.append([f"({lo:.1f},{hi:.1f}]", int(sel.sum()), acc_b, conf_b, abs(acc_b - conf_b)])

    tab = pd.DataFrame(rows, columns=["bin", "count", "acc", "mean_conf", "|acc-conf|"])
    return float(ece), tab

# ---------------------------
# IMPLEMENTATION (runs now)
# ---------------------------
HUMAN_COL = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
AI_ARGMAX_COL = "ai_sent5_from_probs"
PMAX_COL = "ai_pmax"

# Normalize once
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)
df[AI_ARGMAX_COL] = df[AI_ARGMAX_COL].map(norm5)

print("\n==============================")
print("EXTRA LABEL EVAL DIAGNOSTICS")
print("==============================")

# (A) Marginal distributions + JS distance
for pred_col, name in [(AI_HARD_COL, "AI hard"), (AI_ARGMAX_COL, "AI prob-argmax")]:
    tab_h = label_distribution(df[HUMAN_COL])
    tab_a = label_distribution(df[pred_col])
    js = jensen_shannon(tab_h["prop"].values, tab_a["prop"].values)

    print(f"\n--- Marginal shift: Human vs {name} ---")
    display(pd.concat({"Human": tab_h, name: tab_a}, axis=1))
    print("JS distance:", round(js, 4))

# (B) Ordinal-distance error for hard labels (interpretable)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    out = ordinal_error_summary(df[HUMAN_COL], df[pred_col])
    print(f"\n--- Ordinal error: {name} ---")
    for k, v in out.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# (C) Bootstrap CI for accuracy + QWK (hard agreement)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    ci = bootstrap_ci_acc_qwk(df[HUMAN_COL], df[pred_col], B=2000, seed=123)
    print(f"\n--- Bootstrap 95% CI: {name} ---")
    print("N:", ci["N"])
    print(f"Accuracy: {ci['acc_mean']:.4f}  (95% CI {ci['acc_ci95'][0]:.4f}, {ci['acc_ci95'][1]:.4f})")
    print(f"QWK:      {ci['qwk_mean']:.4f}  (95% CI {ci['qwk_ci95'][0]:.4f}, {ci['qwk_ci95'][1]:.4f})")

# (D) Calibration via ECE (for prob-argmax vs human)
if PMAX_COL in df.columns:
    mask = df[HUMAN_COL].notna() & df[AI_ARGMAX_COL].notna() & df[PMAX_COL].notna()
    ece, tab = ece_from_pmax(df.loc[mask, HUMAN_COL],
                             df.loc[mask, AI_ARGMAX_COL],
                             df.loc[mask, PMAX_COL],
                             n_bins=10)
    print("\n--- Calibration: ECE (prob-argmax using pmax) ---")
    print("ECE:", round(ece, 4))
    display(tab)
else:
    print("\n[Skip] ai_pmax not found; cannot compute ECE.")


EXTRA LABEL EVAL DIAGNOSTICS

--- Marginal shift: Human vs AI hard ---


Human           AI hard          
               count      prop   count      prop
very_negative   3953  0.137472    3721  0.129404
negative        1635  0.056860    3240  0.112676
neutral         2282  0.079360    4169  0.144983
positive        5698  0.198157    8681  0.301895
very_positive  15187  0.528152    8944  0.311042

JS distance: 0.171

--- Marginal shift: Human vs AI prob-argmax ---


Human           AI prob-argmax          
               count      prop          count      prop
very_negative   3953  0.137472           4960  0.172492
negative        1635  0.056860           3238  0.112607
neutral         2282  0.079360            796  0.027682
positive        5698  0.198157           9743  0.338828
very_positive  15187  0.528152          10018  0.348392

JS distance: 0.1754

--- Ordinal error: Human vs AI hard ---
N: 28755
mean_abs_ord_err: 0.5465
ord_rmse: 0.8750
within_1_class: 0.9232

--- Ordinal error: Human vs AI prob-argmax ---
N: 28755
mean_abs_ord_err: 0.5454
ord_rmse: 0.9241
within_1_class: 0.9156

--- Bootstrap 95% CI: Human vs AI hard ---
N: 28755
Accuracy: 0.5457  (95% CI 0.5399, 0.5515)
QWK:      0.8121  (95% CI 0.8072, 0.8167)

--- Bootstrap 95% CI: Human vs AI prob-argmax ---
N: 28755
Accuracy: 0.5713  (95% CI 0.5657, 0.5768)
QWK:      0.8053  (95% CI 0.7999, 0.8105)

--- Calibration: ECE (prob-argmax using pmax) ---
ECE: 0.0985


,bin,count,acc,mean_conf,|acc-conf|
0,"(0.0,0.1]",0,NaN,NaN,NaN
1,"(0.1,0.2]",1,0.000000,0.200000,0.200000
2,"(0.2,0.3]",2310,0.230736,0.299681,0.068945
3,"(0.3,0.4]",8839,0.372214,0.390294,0.018080
4,"(0.4,0.5]",9931,0.633874,0.492983,0.140891
5,"(0.5,0.6]",4154,0.796341,0.598577,0.197764
6,"(0.6,0.7]",1324,0.811178,0.695141,0.116037
7,"(0.7,0.8]",1306,0.876723,0.796842,0.079881
8,"(0.8,0.9]",735,0.884354,0.898758,0.014404
9,"(0.9,1.0]",155,0.851613,1.000000,0.148387


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# Helpers
# ---------------------------
ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
PROB_COLS_DEFAULT = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(P, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    rs = P.sum(axis=1, keepdims=True)
    return P / rs

def entropy_norm(P):
    # normalized Shannon entropy in [0,1]
    K = P.shape[1]
    H = -np.sum(P * np.log(P), axis=1)
    return H / np.log(K)

def fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    prob_cols=PROB_COLS_DEFAULT,
    use_entropy=True,                 # if False, uses pmax instead
    add_eff_saf=False,                # optionally add efficacy/safety if present
    robust="HC3",                     # "HC3" (default) or None
    cluster_col=None,                 # e.g., "drugName" if you want cluster-robust SEs
):
    d = df.copy()

    # --- normalize labels ---
    d[human_col] = d[human_col].map(norm5)
    d[ai_col] = d[ai_col].map(norm5)

    # --- keep rows with both labels ---
    mask = d[human_col].notna() & d[ai_col].notna()
    d = d.loc[mask].copy()

    # --- outcome: disagreement indicator ---
    d["disagree"] = (d[human_col] != d[ai_col]).astype(int)

    # --- review length ---
    if review_col in d.columns:
        d["review_len"] = d[review_col].astype(str).str.len()
    else:
        d["review_len"] = np.nan

    # --- rating numeric ---
    if rating_col in d.columns:
        d["rating_num"] = pd.to_numeric(d[rating_col], errors="coerce")
    else:
        d["rating_num"] = np.nan

    # --- uncertainty features from probabilities ---
    missing = [c for c in prob_cols if c not in d.columns]
    if missing:
        raise ValueError(f"Missing prob columns needed for uncertainty: {missing}")

    P = d[prob_cols].astype(float).to_numpy()
    P = defensive_renorm(P)
    d["pmax"] = P.max(axis=1)
    d["entropy_norm"] = entropy_norm(P)

    # --- choose uncertainty regressor ---
    unc = "entropy_norm" if use_entropy else "pmax"

    # --- build formula ---
    base_terms = [unc, "review_len", "rating_num"]

    # optionally add efficacy/safety as categorical controls if columns exist
    if add_eff_saf:
        for col in ["ai_efficacy", "ai_safety"]:
            if col in d.columns:
                base_terms.append(f"C({col})")

    # drop rows with missing covariates used in model
    needed_cols = ["disagree", unc, "review_len", "rating_num"]
    d_model = d.dropna(subset=needed_cols).copy()

    formula = "disagree ~ " + " + ".join(base_terms)

    # --- fit logit ---
    model = smf.logit(formula, data=d_model)
    if cluster_col is not None and cluster_col in d_model.columns:
        res = model.fit(disp=False, cov_type="cluster", cov_kwds={"groups": d_model[cluster_col]})
    elif robust is not None:
        res = model.fit(disp=False, cov_type=robust)
    else:
        res = model.fit(disp=False)

    return res, formula, d_model


def summarize_odds_ratios(res):
    """Pretty OR table with 95% CI."""
    params = res.params
    se = res.bse
    z = 1.96
    out = pd.DataFrame({
        "coef": params,
        "se": se,
        "OR": np.exp(params),
        "OR_2.5%": np.exp(params - z*se),
        "OR_97.5%": np.exp(params + z*se),
        "p": res.pvalues
    })
    return out.sort_values("p")


# ---------------------------
# IMPLEMENTATION ON df
# ---------------------------

# 1) Disagreement ~ entropy + controls
res_ent, formula_ent, d_used_ent = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=True,
    add_eff_saf=False,   # flip to True if you want to control for ai_efficacy/ai_safety
    robust="HC3",
    cluster_col=None,    # e.g. "drugName" if you want clustered SE
)
print("MODEL (entropy):", formula_ent)
print(res_ent.summary())
display(summarize_odds_ratios(res_ent).head(15))

# 2) Disagreement ~ pmax + controls
res_pmax, formula_pmax, d_used_pmax = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=False,   # uses pmax
    add_eff_saf=False,
    robust="HC3",
    cluster_col=None,
)
print("\nMODEL (pmax):", formula_pmax)
print(res_pmax.summary())
display(summarize_odds_ratios(res_pmax).head(15))

MODEL (entropy): disagree ~ entropy_norm + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1850
Time:                        23:46:41   Log-Likelihood:                -16146.
converged:                       True   LL-Null:                       -19811.
Covariance Type:                  HC3   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -5.8308      0.088    -66.167      0.000      -6.004      -5.658
entropy_norm     6.8595      0.100     68.280      0.000  

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.830820,0.088122,0.002936,0.002470,0.003489,0.000000e+00
entropy_norm,6.859527,0.100461,952.916648,782.600979,1160.297729,0.000000e+00
rating_num,0.182447,0.004795,1.200151,1.188923,1.211484,0.000000e+00
review_len,-0.000742,0.000058,0.999258,0.999144,0.999372,3.126739e-37



MODEL (pmax): disagree ~ pmax + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1816
Time:                        23:46:41   Log-Likelihood:                -16214.
converged:                       True   LL-Null:                       -19811.
Covariance Type:                  HC3   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.5130      0.079     44.500      0.000       3.358       3.668
pmax          -9.5434      0.155    -61.686      0.000      -9.847      -9

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.512965,0.078942,33.547577,28.738496,39.161406,0.000000e+00
pmax,-9.543417,0.154711,0.000072,0.000053,0.000097,0.000000e+00
rating_num,0.159376,0.004594,1.172779,1.162266,1.183387,1.122222e-263
review_len,-0.000573,0.000057,0.999427,0.999317,0.999538,5.126214e-24


In [ ]:
import numpy as np
import pandas as pd

# ---- config ----
HUMAN_RATING = "rating"          # original user rating column (string/int)
HUMAN5 = "sentiment_5"           # human 5-class
AI5 = "ai_sentiment_5"           # AI hard 5-class
AI_R10 = "ai_rating_10"          # AI 1..10

def norm5(x):
    if pd.isna(x): return np.nan
    return str(x).strip().lower().replace(" ", "_").replace("-", "_")

def rating10_to_sent5(r):
    if pd.isna(r): return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

# =========================
# 1) Check human rating_num
# =========================
df["rating_num"] = pd.to_numeric(df[HUMAN_RATING], errors="coerce")

print("Human rating_num summary:")
print(df["rating_num"].describe())
print("\nHuman rating_num invalid (NaN or outside 1..10):",
      int(df["rating_num"].isna().sum() + ((df["rating_num"]<1)|(df["rating_num"]>10)).sum()))
print("\nTop raw values of `rating` (to confirm it's the original column):")
print(df[HUMAN_RATING].value_counts(dropna=False).head(15))

# If you have a `date` column and want to check parsing issues:
if "date" in df.columns:
    print("\nDate col dtype:", df["date"].dtype)

# ======================
# 2) Check AI ai_rating_10
# ======================
df["ai_rating10_num"] = pd.to_numeric(df[AI_R10], errors="coerce")

print("\nAI ai_rating_10 summary:")
print(df["ai_rating10_num"].describe())
print("\nAI ai_rating_10 invalid (NaN or outside 1..10):",
      int(df["ai_rating10_num"].isna().sum() + ((df["ai_rating10_num"]<1)|(df["ai_rating10_num"]>10)).sum()))
print("\nTop raw values of `ai_rating_10`:")
print(df[AI_R10].value_counts(dropna=False).head(15))

# ==========================================
# 3) Check ai_sentiment_5 <-> ai_rating_10
# ==========================================
df["ai_sent5_from_rating10"] = df["ai_rating10_num"].apply(rating10_to_sent5)
df["ai_sentiment_5_clean"] = df[AI5].map(norm5)

mask = df["ai_sent5_from_rating10"].notna() & df["ai_sentiment_5_clean"].notna()
match_rate = (df.loc[mask, "ai_sent5_from_rating10"] == df.loc[mask, "ai_sentiment_5_clean"]).mean()

print("\nAI hard (ai_sentiment_5) vs binned ai_rating_10 match rate:", round(match_rate, 6))
if match_rate < 1.0:
    bad = df.loc[mask].copy()
    bad = bad[bad["ai_sent5_from_rating10"] != bad["ai_sentiment_5_clean"]]
    print("Mismatches:", len(bad))
    display(bad[[AI_R10, "ai_sent5_from_rating10", AI5, "review"]].head(20))

# ==========================================
# 4) Confirm disagreement outcome definition
# ==========================================
df["human_sent5_clean"] = df[HUMAN5].map(norm5)
mask2 = df["human_sent5_clean"].notna() & df["ai_sentiment_5_clean"].notna()

df["disagree"] = (df["human_sent5_clean"] != df["ai_sentiment_5_clean"]).astype(int)

print("\nDisagreement rate (human 5 vs AI hard 5):",
      df.loc[mask2, "disagree"].mean().round(4),
      "| N =", int(mask2.sum()))

# quick crosstab sanity
print("\nCrosstab (human vs AI hard) top-left sanity:")
display(pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"]).reindex(
    index=["very_negative","negative","neutral","positive","very_positive"],
    columns=["very_negative","negative","neutral","positive","very_positive"]
))

Human rating_num summary:
count    28755.000000
mean         7.407060
std          3.037296
min          1.000000
25%          6.000000
50%          9.000000
75%         10.000000
max         10.000000
Name: rating_num, dtype: float64

Human rating_num invalid (NaN or outside 1..10): 0

Top raw values of `rating` (to confirm it's the original column):
rating
10    9444
9     5743
8     3873
1     2944
7     1825
5     1163
6     1119
2     1009
3      927
4      708
Name: count, dtype: int64

Date col dtype: object

AI ai_rating_10 summary:
count    28755.000000
mean         6.614050
std          2.705521
min          1.000000
25%          5.000000
50%          7.000000
75%          9.000000
max         10.000000
Name: ai_rating10_num, dtype: float64

AI ai_rating_10 invalid (NaN or outside 1..10): 0

Top raw values of `ai_rating_10`:
ai_rating_10
9.0     5875
8.0     5201
7.0     3480
10.0    3069
6.0     2418
2.0     2283
5.0     1751
4.0     1667
3.0     1573
1.0     1438
Name: coun

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,2944,869,114,19,7
negative,466,859,264,36,10
neutral,190,862,964,235,31
positive,73,417,1798,2720,690
very_positive,48,233,1029,5671,8206


In [ ]:
ct = pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"])
ct = ct.reindex(index=["very_negative","negative","neutral","positive","very_positive"],
                columns=["very_negative","negative","neutral","positive","very_positive"],
                fill_value=0)
display(ct)

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,2944,869,114,19,7
negative,466,859,264,36,10
neutral,190,862,964,235,31
positive,73,417,1798,2720,690
very_positive,48,233,1029,5671,8206


In [ ]:
PROB_COLS = ["ai_prob_very_negative","ai_prob_negative","ai_prob_neutral","ai_prob_positive","ai_prob_very_positive"]
P = df[PROB_COLS].astype(float)

# any negatives?
print("Any prob < 0:", (P < 0).any().any())

row_sum = P.sum(axis=1)
print("Row-sum summary:")
print(row_sum.describe())
print("Rows far from 1.0 (abs > 1e-3):", int((row_sum.sub(1).abs() > 1e-3).sum()))

Any prob < 0: False
Row-sum summary:
count    2.875500e+04
mean     1.000000e+00
std      1.335630e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
Rows far from 1.0 (abs > 1e-3): 0


In [ ]:
pmax = P.max(axis=1)
entropy = -(P.clip(1e-12,1).to_numpy() * np.log(P.clip(1e-12,1).to_numpy())).sum(axis=1) / np.log(5)

print("corr(entropy_norm, pmax) =", np.corrcoef(entropy, pmax)[0,1])

corr(entropy_norm, pmax) = -0.9353152201515607


In [ ]:
import numpy as np
import pandas as pd

HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x): return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def make_df2(df):
    df2 = df.copy()

    # --- clean labels ---
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sentiment_5_clean"] = df2[AI_HARD_COL].map(norm5)

    # --- outcome: disagree (hard vs human) ---
    mask = df2["human_sent5_clean"].notna() & df2["ai_sentiment_5_clean"].notna()
    df2.loc[mask, "disagree"] = (df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sentiment_5_clean"]).astype(int)

    # --- predictors: review length ---
    if "review" in df2.columns:
        df2["review_len"] = df2["review"].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # --- predictors: rating numeric (from human rating column "rating") ---
    # If you already have rating_num, this will just overwrite consistently.
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif "rating" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating"], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # --- predictors: pmax + entropy_norm from probability vector ---
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if len(missing) > 0:
        raise KeyError(f"Missing probability columns needed for entropy/pmax: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    return df2

df2 = make_df2(df)
print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

N (with disagree defined): 28755
Disagree rate: 0.45425143453312467


In [ ]:
# ============================================================
# FULL DROP-IN CELL: Rescaled Logit + robust SE via cov_type="HC3"
# - ORs are per 0.1 increase in entropy/pmax
# - ORs are per 100 characters increase in review length
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# CONFIG (edit if needed)
# ---------------------------
HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
REVIEW_COL  = "review"
RATING_COL  = "rating"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

# ---------------------------
# HELPERS
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    s = p.sum(axis=1, keepdims=True)
    return p / s

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def build_df2(df):
    df2 = df.copy()

    # clean labels
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sent5_clean"]    = df2[AI_HARD_COL].map(norm5)

    # outcome: disagree (based on HUMAN vs AI HARD labels)
    mask = df2["human_sent5_clean"].notna() & df2["ai_sent5_clean"].notna()
    df2.loc[mask, "disagree"] = (
        df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sent5_clean"]
    ).astype(int)

    # review length
    if REVIEW_COL in df2.columns:
        df2["review_len"] = df2[REVIEW_COL].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # rating numeric
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif RATING_COL in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2[RATING_COL], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # probs -> pmax + entropy_norm
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if missing:
        raise KeyError(f"Missing probability columns: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    # ---------------------------
    # RESCALES for interpretability
    # ---------------------------
    # OR per 0.1 increase (instead of per 1.0)
    df2["entropy_per_0p1"] = df2["entropy_norm"] / 0.1
    df2["pmax_per_0p1"]    = df2["pmax"] / 0.1

    # OR per 100 characters (instead of per 1 char)
    df2["review_len_per_100"] = df2["review_len"] / 100.0

    return df2

def fit_disagree_logit(df2, predictor, cov_type="HC3"):
    """
    Fit: disagree ~ predictor + review_len_per_100 + rating_num
    """
    needed = ["disagree", predictor, "review_len_per_100", "rating_num"]
    d = df2[needed].dropna().copy()
    d["disagree"] = d["disagree"].astype(int)

    formula = f"disagree ~ {predictor} + review_len_per_100 + rating_num"
    res = smf.logit(formula, data=d).fit(disp=False, cov_type=cov_type)
    return res, d

def or_table_from_result(res, alpha=0.05):
    b = res.params
    se = res.bse
    p = res.pvalues
    ci = res.conf_int(alpha=alpha)
    ci.columns = ["coef_2.5%", "coef_97.5%"]

    out = pd.DataFrame({
        "coef": b,
        "se": se,
        "OR": np.exp(b),
        "OR_2.5%": np.exp(ci["coef_2.5%"]),
        "OR_97.5%": np.exp(ci["coef_97.5%"]),
        "p": p,
    })
    return out

# ============================================================
# RUN (ASSUMES df exists)
# ============================================================

df2 = build_df2(df)

print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

# ---- Model 1 (RESCALED): entropy_per_0p1 ----
res_ent, _ = fit_disagree_logit(df2, predictor="entropy_per_0p1", cov_type="HC3")
print("\nMODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num")
print(res_ent.summary())
display(or_table_from_result(res_ent))

# ---- Model 2 (RESCALED): pmax_per_0p1 ----
res_pmax, _ = fit_disagree_logit(df2, predictor="pmax_per_0p1", cov_type="HC3")
print("\nMODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num")
print(res_pmax.summary())
display(or_table_from_result(res_pmax))

N (with disagree defined): 28755
Disagree rate: 0.45425143453312467

MODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1850
Time:                        23:46:42   Log-Likelihood:                -16146.
converged:                       True   LL-Null:                       -19811.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -5.8308      0.

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.830820,0.088122,0.002936,0.002470,0.003489,0.000000e+00
entropy_per_0p1,0.685953,0.010046,1.985663,1.946947,2.025148,0.000000e+00
review_len_per_100,-0.074234,0.005822,0.928454,0.917919,0.939110,3.126739e-37
rating_num,0.182447,0.004795,1.200151,1.188923,1.211484,0.000000e+00



MODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1816
Time:                        23:46:42   Log-Likelihood:                -16214.
converged:                       True   LL-Null:                       -19811.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              3.5130      0.079     44.500      0.000       3.358       3.668
pmax_per_0p1          -0

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.512965,0.078942,33.547577,28.738578,39.161295,0.000000e+00
pmax_per_0p1,-0.954342,0.015471,0.385066,0.373565,0.396921,0.000000e+00
review_len_per_100,-0.057267,0.005666,0.944342,0.933913,0.954887,5.126214e-24
rating_num,0.159376,0.004594,1.172779,1.162266,1.183387,1.122222e-263


In [2]:
import pandas as pd
import numpy as np
import re

def inspect_review_duplicates(df, review_col="review", top_k=20, show_examples=3, clean_ws=True):
    d = df.copy()

    # 1) Basic missingness
    na_n = int(d[review_col].isna().sum()) if review_col in d.columns else None
    print(f"Column: {review_col}")
    print(f"N rows: {len(d)}")
    print(f"Missing reviews: {na_n}")

    # 2) Choose raw vs cleaned key for grouping
    if clean_ws:
        key = (d[review_col]
               .fillna("")
               .astype(str)
               .str.replace(r"\s+", " ", regex=True)   # collapse whitespace
               .str.strip())
        key_name = "review_key_cleanws"
    else:
        key = d[review_col].fillna("").astype(str)
        key_name = "review_key_raw"

    d[key_name] = key

    # Watch out for literal "nan" strings (common artifact if someone did astype(str) earlier)
    nan_string_n = int((d[key_name].str.lower() == "nan").sum())
    empty_n = int((d[key_name] == "").sum())
    print(f'Empty string reviews after keying: {empty_n}')
    print(f'Literal "nan" string reviews after keying: {nan_string_n}')

    # 3) Duplicate counts
    vc = d[key_name].value_counts(dropna=False)
    unique_texts = int(vc.shape[0])
    dup_groups = int((vc >= 2).sum())
    dup_instances = int(vc[vc >= 2].sum())
    dup_pct = dup_instances / len(d)

    print("\n=== Duplicate summary (by exact match on key) ===")
    print(f"unique texts: {unique_texts}")
    print(f"duplicate groups (count>=2): {dup_groups}")
    print(f"duplicate instances (in those groups): {dup_instances}  ({dup_pct:.3%})")
    print(f"max duplicate group size: {int(vc.max())}")

    # 4) Show top duplicate groups + example texts
    top = vc[vc >= 2].head(top_k)
    if top.empty:
        print("\nNo duplicates found under this key definition.")
        return

    print(f"\n=== Top {min(top_k, len(top))} duplicate groups ===")
    display(top.to_frame("count"))

    print("\n=== Example duplicate texts ===")
    for i, (txt, cnt) in enumerate(top.items(), start=1):
        print(f"\n[{i}] count = {cnt}")
        # show a snippet so it doesn’t flood output
        snippet = txt[:500] + (" ..." if len(txt) > 500 else "")
        print(snippet)

        # show a few row indices where it occurs
        idxs = d.index[d[key_name] == txt].tolist()[:show_examples]
        print(f"example row indices: {idxs}")

def compare_raw_vs_cleanws(df, review_col="review"):
    print("---- RAW grouping ----")
    inspect_review_duplicates(df, review_col=review_col, clean_ws=False, top_k=10, show_examples=2)
    print("\n\n---- CLEANED whitespace grouping ----")
    inspect_review_duplicates(df, review_col=review_col, clean_ws=True, top_k=10, show_examples=2)

# ====== RUN ONE of these ======
# 1) single view (cleaned whitespace by default)
# inspect_review_duplicates(df, review_col="review", clean_ws=True)

# 2) compare raw vs cleaned grouping (recommended)
compare_raw_vs_cleanws(df, review_col="review")

---- RAW grouping ----
Column: review
N rows: 28755
Missing reviews: 0
Empty string reviews after keying: 0
Literal "nan" string reviews after keying: 0

=== Duplicate summary (by exact match on key) ===
unique texts: 17732
duplicate groups (count>=2): 10984
duplicate instances (in those groups): 22007  (76.533%)
max duplicate group size: 9

=== Top 10 duplicate groups ===


,count
review_key_raw,
"""Good""",9
"""Good.""",6
"""This med is amazing. 23 year old christian college grad with every type of anxiety for the past 2 years. Could barely go to class or the grocery store, felt dissociated, distant and terrified of people (including friends). Was scared to try any drug but after losing my job and needing to go out and interview I decided to try something. 3 days in and I feel totally changed. No anxiety. Clear head. Feel like I could even move countries to work and I&#039;ve been out more in a weekend than I have been in 3 months combined. I take 5mg and 25mg of hydroxyzine to balance out the side effects (blurry vision, headaches, muscle spasms etc). Don&#039;t be scared just try it. I regret not going on this sooner.""",4
"""I love it.""",4
"""Best medicine for anxiety.""",4
"""For several years I have been prescribed 1mg to take up to 3 times per day - sometimes I take less, some days even none! But I find that Xanax does for me what adderall does for children suffering add. For some reason the drug helps me have precision focus, a clear mind, better communication, rapid fire thought processing, and motivation - just for starters. I don&#039;t expect to take it for the rest of my life, but I go thro prolonged periods of anxiety and antisocial behavior (will urinate in bottles in my room when at my worst). Glad to know there are a few people who Experience the same effects using Xanax as Maintenance. It&#039;s been a life saver for me! Curious to understand why it has this off label effect on us select few.""",4
"""It works really well.""",4
"""Works great for anxiety.""",4
"""Most of my life I have struggled with severe panic attacks, general anxiety, PTSD &amp; moderate depression. Been on meds twice before until I felt better &amp; then I would stop taking them. This last time it had been close to a year that I had been suffering again before I decided to reluctantly go back on meds. I tried Celexa again- it worked kinda okay in the past but this time it made no improvement for me. Saw the doc who switched me to Prozac- all I can say is WOW! Best yet- I feel like my old, happy self after only 1 month! 10mg the first week, 15 the second and 20mg the last two. Give it time to work &amp; be patient- it might make you feel a bit off &amp; not better at first, but it is a miracle drug. These bad feelings will pass hang in there!""",4



=== Example duplicate texts ===

[1] count = 9
"Good"
example row indices: [12392, 12947]

[2] count = 6
"Good."
example row indices: [2161, 7183]

[3] count = 4
"This med is amazing. 23 year old christian college grad with every type of anxiety for the past 2 years. Could barely go to class or the grocery store, felt dissociated, distant and terrified of people (including friends). Was scared to try any drug but after losing my job and needing to go out and interview I decided to try something. 3 days in and I feel totally changed. No anxiety. Clear head. Feel like I could even move countries to work and I&#039;ve been out more in a weekend than I have  ...
example row indices: [3431, 19096]

[4] count = 4
"I love it."
example row indices: [3675, 10773]

[5] count = 4
"Best medicine for anxiety."
example row indices: [2758, 6904]

[6] count = 4
"For several years I have been prescribed 1mg to take up to 3 times per day - sometimes I take less, some days even none! But I find that Xan

,count
review_key_cleanws,
"""Good""",9
"""Good.""",6
"""This med is amazing. 23 year old christian college grad with every type of anxiety for the past 2 years. Could barely go to class or the grocery store, felt dissociated, distant and terrified of people (including friends). Was scared to try any drug but after losing my job and needing to go out and interview I decided to try something. 3 days in and I feel totally changed. No anxiety. Clear head. Feel like I could even move countries to work and I&#039;ve been out more in a weekend than I have been in 3 months combined. I take 5mg and 25mg of hydroxyzine to balance out the side effects (blurry vision, headaches, muscle spasms etc). Don&#039;t be scared just try it. I regret not going on this sooner.""",4
"""I love it.""",4
"""Best medicine for anxiety.""",4
"""For several years I have been prescribed 1mg to take up to 3 times per day - sometimes I take less, some days even none! But I find that Xanax does for me what adderall does for children suffering add. For some reason the drug helps me have precision focus, a clear mind, better communication, rapid fire thought processing, and motivation - just for starters. I don&#039;t expect to take it for the rest of my life, but I go thro prolonged periods of anxiety and antisocial behavior (will urinate in bottles in my room when at my worst). Glad to know there are a few people who Experience the same effects using Xanax as Maintenance. It&#039;s been a life saver for me! Curious to understand why it has this off label effect on us select few.""",4
"""It works really well.""",4
"""Works great for anxiety.""",4
"""Most of my life I have struggled with severe panic attacks, general anxiety, PTSD &amp; moderate depression. Been on meds twice before until I felt better &amp; then I would stop taking them. This last time it had been close to a year that I had been suffering again before I decided to reluctantly go back on meds. I tried Celexa again- it worked kinda okay in the past but this time it made no improvement for me. Saw the doc who switched me to Prozac- all I can say is WOW! Best yet- I feel like my old, happy self after only 1 month! 10mg the first week, 15 the second and 20mg the last two. Give it time to work &amp; be patient- it might make you feel a bit off &amp; not better at first, but it is a miracle drug. These bad feelings will pass hang in there!""",4



=== Example duplicate texts ===

[1] count = 9
"Good"
example row indices: [12392, 12947]

[2] count = 6
"Good."
example row indices: [2161, 7183]

[3] count = 4
"This med is amazing. 23 year old christian college grad with every type of anxiety for the past 2 years. Could barely go to class or the grocery store, felt dissociated, distant and terrified of people (including friends). Was scared to try any drug but after losing my job and needing to go out and interview I decided to try something. 3 days in and I feel totally changed. No anxiety. Clear head. Feel like I could even move countries to work and I&#039;ve been out more in a weekend than I have  ...
example row indices: [3431, 19096]

[4] count = 4
"I love it."
example row indices: [3675, 10773]

[5] count = 4
"Best medicine for anxiety."
example row indices: [2758, 6904]

[6] count = 4
"For several years I have been prescribed 1mg to take up to 3 times per day - sometimes I take less, some days even none! But I find that Xan

In [6]:
import numpy as np
import pandas as pd

# ----------------------------
# Utilities
# ----------------------------
def make_review_key(df, review_col="review", clean_ws=True, key_col="review_key"):
    d = df.copy()
    if review_col not in d.columns:
        raise KeyError(f"review_col='{review_col}' not found. Available cols: {list(d.columns)[:30]} ...")
    s = d[review_col].fillna("").astype(str)
    if clean_ws:
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    d[key_col] = s
    return d

def guess_label_cols(df):
    # tries to find plausible label columns automatically
    candidates = []
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ["label", "rating", "score", "sentiment", "y_", "target"]):
            candidates.append(c)
    # remove obviously-non-label columns
    bad = set(["split", "id", "review", "text", "text_orig"])
    candidates = [c for c in candidates if c.lower() not in bad]
    return candidates

def _maj_share(s):
    vc = s.value_counts(dropna=False)
    return float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan

def _entropy(s):
    vc = s.value_counts(dropna=False, normalize=True)
    p = vc.values
    return float(-(p * np.log(p + 1e-12)).sum())

# ----------------------------
# 1) Duplicate label consistency
# ----------------------------
def dup_label_consistency(
    df,
    review_col="review",
    label_cols=None,
    clean_ws=True,
    key_col="review_key",
):
    d = make_review_key(df, review_col=review_col, clean_ws=clean_ws, key_col=key_col)

    if label_cols is None:
        label_cols = guess_label_cols(d)
        if not label_cols:
            raise ValueError(
                "Could not auto-detect label columns. Pass label_cols=[...] explicitly."
            )

    missing = [c for c in label_cols if c not in d.columns]
    if missing:
        raise KeyError(f"label_cols missing in df: {missing}. Available cols: {list(d.columns)[:30]} ...")

    g = d.groupby(key_col, dropna=False)
    out = pd.DataFrame({"n": g.size()})

    for col in label_cols:
        out[f"{col}_nunique"] = g[col].nunique(dropna=False)
        out[f"{col}_majshare"] = g[col].apply(_maj_share)
        out[f"{col}_entropy"]  = g[col].apply(_entropy)

    out = out.reset_index()
    dup = out[out["n"] >= 2].copy()
    return d, out, dup, label_cols

# ----------------------------
# 2) Text-only ceiling for each label source
# ----------------------------
def text_only_ceiling(df, key_col, label_col):
    g = df.groupby(key_col, dropna=False)[label_col]
    best_correct = g.apply(lambda s: s.value_counts(dropna=False).max()).sum()
    return float(best_correct / len(df))

# ----------------------------
# 3) Majority label agreement between two sources (per unique text)
# ----------------------------
def group_majority(df, key_col, label_col):
    return df.groupby(key_col, dropna=False)[label_col].apply(
        lambda s: s.value_counts(dropna=False).idxmax()
    )

def majority_agreement(df, key_col, label_a, label_b):
    a = group_majority(df, key_col, label_a).rename(label_a)
    b = group_majority(df, key_col, label_b).rename(label_b)
    maj = pd.concat([a, b], axis=1).dropna()
    return float((maj[label_a] == maj[label_b]).mean()), int(len(maj))

# ----------------------------
# 4) Metadata variation within duplicate texts (optional)
# ----------------------------
def dup_metadata_variation(df, key_col, drug_col="drug_id", cond_col="cond_id", useful_col="useful_z"):
    for c in [drug_col, cond_col, useful_col]:
        if c not in df.columns:
            raise KeyError(f"'{c}' not found in df. Available cols: {list(df.columns)[:30]} ...")

    g = df.groupby(key_col, dropna=False)
    out = g.agg(
        n=(key_col, "size"),
        drug_nunique=(drug_col, "nunique"),
        cond_nunique=(cond_col, "nunique"),
        useful_nunique=(useful_col, "nunique"),
    ).reset_index()

    out = out[out["n"] >= 2].copy()
    out["drug_or_cond_varies"] = (out["drug_nunique"] > 1) | (out["cond_nunique"] > 1)
    return out

# ==========================================================
# RUN THIS SECTION (edit only review_col / label_cols if needed)
# ==========================================================

# 1) Build keys + duplicate consistency table
d, all_groups, dup_groups, detected_labels = dup_label_consistency(
    df,
    review_col="review",       # <-- change if your column isn't named "review"
    label_cols=None,           # <-- OR set explicitly, e.g. ["human_label","gpt_text_label","gpt_meta_label"]
    clean_ws=True,
    key_col="review_key_cleanws",
)

print("Detected label columns:", detected_labels)
print("\nDuplicate rate (by cleaned key):",
      (d["review_key_cleanws"].duplicated().mean()))

print("\nSummary over duplicate groups (n>=2):")
display(dup_groups.describe(include="all"))

# 2) Text-only ceilings for each detected label column
print("\n=== Text-only ceiling (upper bound) by label source ===")
for col in detected_labels:
    try:
        ce = text_only_ceiling(d, "review_key_cleanws", col)
        print(f"{col}: {ce:.4f}")
    except Exception as e:
        print(f"{col}: [skip] {e}")

# 3) Pairwise majority agreement across label sources (per unique text)
print("\n=== Majority agreement per unique text (pairwise) ===")
for i in range(len(detected_labels)):
    for j in range(i+1, len(detected_labels)):
        a, b = detected_labels[i], detected_labels[j]
        agree, n_texts = majority_agreement(d, "review_key_cleanws", a, b)
        print(f"{a} vs {b}: agreement={agree:.4f} over {n_texts} unique texts")

# 4) OPTIONAL: metadata variation within duplicate text (only if these cols exist)
meta_cols = {"drug_id", "cond_id", "useful_z"}
if meta_cols.issubset(set(d.columns)):
    mv = dup_metadata_variation(d, "review_key_cleanws", "drug_id", "cond_id", "useful_z")
    print("\n=== Duplicate texts: how often metadata differs? ===")
    print("Share of duplicate groups where drug or condition varies:",
          float(mv["drug_or_cond_varies"].mean()))
else:
    print("\n[Info] Skipping metadata-variation check: need columns drug_id, cond_id, useful_z.")

Detected label columns: ['rating', 'sentiment_5', 'ai_rating_10', 'ai_sentiment_5', 'ai_prob_very_negative', 'ai_prob_very_positive', 'review_key_cleanws']

Duplicate rate (by cleaned key): 0.3833420274734829

Summary over duplicate groups (n>=2):


,review_key_cleanws,n,rating_nunique,rating_majshare,rating_entropy,sentiment_5_nunique,sentiment_5_majshare,sentiment_5_entropy,ai_rating_10_nunique,ai_rating_10_majshare,...,ai_sentiment_5_entropy,ai_prob_very_negative_nunique,ai_prob_very_negative_majshare,ai_prob_very_negative_entropy,ai_prob_very_positive_nunique,ai_prob_very_positive_majshare,ai_prob_very_positive_entropy,review_key_cleanws_nunique,review_key_cleanws_majshare,review_key_cleanws_entropy
count,10984,10984.000000,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,...,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.0,10984.0,1.098400e+04
unique,10984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,"""works very well.""",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.003551,1.001001,0.999686,5.519155e-04,1.000455,0.999868,2.526629e-04,1.393208,0.803553,...,1.384140e-01,1.336034,0.832044,2.328922e-01,1.652130,0.674610,4.515052e-01,1.0,1.0,-1.000089e-12
std,NaN,0.102265,0.041581,0.011802,2.090934e-02,0.025242,0.007306,1.375501e-02,0.488671,0.244146,...,2.770714e-01,0.472372,0.236130,3.273880e-01,0.480124,0.238401,3.310439e-01,0.0,0.0,7.088711e-26
min,NaN,2.000000,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.333333,-1.000089e-12,1.0,1.0,-1.000089e-12
25%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,0.500000,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.0,1.0,-1.000089e-12
50%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,...,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,0.500000,6.931472e-01,1.0,1.0,-1.000089e-12
75%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,1.000000,...,-1.000089e-12,2.000000,1.000000,6.931472e-01,2.000000,1.000000,6.931472e-01,1.0,1.0,-1.000089e-12



=== Text-only ceiling (upper bound) by label source ===
rating: 0.9995
sentiment_5: 0.9997
ai_rating_10: 0.8496
ai_sentiment_5: 0.9235
ai_prob_very_negative: 0.8716
ai_prob_very_positive: 0.7508
review_key_cleanws: 1.0000

=== Majority agreement per unique text (pairwise) ===
rating vs sentiment_5: agreement=0.0000 over 17732 unique texts
rating vs ai_rating_10: agreement=0.2907 over 17732 unique texts
rating vs ai_sentiment_5: agreement=0.0000 over 17732 unique texts
rating vs ai_prob_very_negative: agreement=0.0000 over 17732 unique texts
rating vs ai_prob_very_positive: agreement=0.0000 over 17732 unique texts
rating vs review_key_cleanws: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_rating_10: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_sentiment_5: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_prob_very_negative: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_prob_very_positive: agreement=0.0000 over 17732 unique texts
sentime